In [1]:
import pandas as pd
import numpy as np
import json
import os

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
file_path = "../data/processed/processed_interactions.csv"

df = pd.read_csv(file_path)

print("Processed dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

Processed dataset loaded successfully.
Shape: (446844, 11)


,interaction_order,student_id,session_id,problem_id,concept_name,correct,attempt_count,hint_count,hint_total,response_time_ms,opportunity
0,21617623,14,263599,93383,Circle Graph,0,1,2,2,26271,1
1,21617623,14,263599,93383,Percent Of,0,1,2,2,26271,1
2,21617632,14,263599,93407,Circle Graph,1,1,0,2,29123,2
3,21617632,14,263599,93407,Percent Of,1,1,0,2,29123,2
4,21617641,14,263599,93400,Circle Graph,0,1,2,2,13779,3


In [4]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nCorrect values:")
print(df["correct"].value_counts())

Columns:
['interaction_order', 'student_id', 'session_id', 'problem_id', 'concept_name', 'correct', 'attempt_count', 'hint_count', 'hint_total', 'response_time_ms', 'opportunity']

Missing values:
interaction_order    0
student_id           0
session_id           0
problem_id           0
concept_name         0
correct              0
attempt_count        0
hint_count           0
hint_total           0
response_time_ms     0
opportunity          0
dtype: int64

Correct values:
correct
1    310640
0    136204
Name: count, dtype: int64


In [6]:
#Select One Sample Student
sample_student_id = 14

student_df = df[df["student_id"] == sample_student_id].copy()

print("Student ID:", sample_student_id)
print("Total interactions:", len(student_df))
print("Sessions:", student_df["session_id"].nunique())
print("Concepts:", student_df["concept_name"].nunique())

student_df.head(10)

Student ID: 14
Total interactions: 35
Sessions: 3
Concepts: 7


,interaction_order,student_id,session_id,problem_id,concept_name,correct,attempt_count,hint_count,hint_total,response_time_ms,opportunity
0,21617623,14,263599,93383,Circle Graph,0,1,2,2,26271,1
1,21617623,14,263599,93383,Percent Of,0,1,2,2,26271,1
2,21617632,14,263599,93407,Circle Graph,1,1,0,2,29123,2
3,21617632,14,263599,93407,Percent Of,1,1,0,2,29123,2
4,21617641,14,263599,93400,Circle Graph,0,1,2,2,13779,3
5,21617641,14,263599,93400,Percent Of,0,1,2,2,13779,3
6,21617650,14,263599,93419,Circle Graph,0,1,2,2,16901,4
7,21617650,14,263599,93419,Percent Of,0,1,2,2,16901,4
8,21617659,14,263599,93420,Circle Graph,0,1,2,2,11079,5
9,21617659,14,263599,93420,Percent Of,0,1,2,2,11079,5


In [7]:
#Create Short-Term Memory (Short-term memory stores the current session context)

latest_session_id = student_df["session_id"].iloc[-1]

session_df = student_df[student_df["session_id"] == latest_session_id].copy()

current_concept = session_df["concept_name"].iloc[-1]
recent_accuracy = session_df["correct"].mean()
recent_attempts = session_df["attempt_count"].mean()
recent_hint_usage = session_df["hint_count"].sum()

if recent_accuracy >= 0.75:
    session_status = "good_progress"
elif recent_accuracy >= 0.50:
    session_status = "moderate_progress"
else:
    session_status = "needs_support"

short_term_memory = {
    "student_id": int(sample_student_id),
    "session_id": int(latest_session_id),
    "current_concept": current_concept,
    "recent_interaction_count": int(len(session_df)),
    "recent_accuracy": round(float(recent_accuracy), 2),
    "average_attempts": round(float(recent_attempts), 2),
    "recent_hint_usage": int(recent_hint_usage),
    "session_status": session_status
}

short_term_memory

{'student_id': 14,
 'session_id': 263601,
 'current_concept': 'Median',
 'recent_interaction_count': 7,
 'recent_accuracy': 0.43,
 'average_attempts': 1.0,
 'recent_hint_usage': 10,
 'session_status': 'needs_support'}

In [8]:
#Create Long-Term Memory (Long-term memory stores persistent student behavior across sessions.)

overall_accuracy = student_df["correct"].mean()
average_attempts = student_df["attempt_count"].mean()
average_hint_count = student_df["hint_count"].mean()
average_response_time = student_df["response_time_ms"].mean()

concept_summary = student_df.groupby("concept_name").agg(
    total_interactions=("correct", "count"),
    correct_count=("correct", "sum"),
    avg_hints=("hint_count", "mean"),
    avg_attempts=("attempt_count", "mean"),
    avg_response_time_ms=("response_time_ms", "mean")
).reset_index()

concept_summary["wrong_count"] = (
    concept_summary["total_interactions"] - concept_summary["correct_count"]
)

concept_summary["accuracy"] = (
    concept_summary["correct_count"] / concept_summary["total_interactions"]
)

weak_areas = concept_summary[
    (concept_summary["accuracy"] < 0.50) &
    (concept_summary["total_interactions"] >= 3)
]["concept_name"].tolist()

strong_areas = concept_summary[
    (concept_summary["accuracy"] >= 0.75) &
    (concept_summary["total_interactions"] >= 3)
]["concept_name"].tolist()

if average_hint_count >= 2:
    preferred_support_style = "guided_support"
elif average_attempts >= 2:
    preferred_support_style = "step_by_step_support"
elif overall_accuracy >= 0.75:
    preferred_support_style = "independent_practice"
else:
    preferred_support_style = "basic_explanation_support"

long_term_memory = {
    "student_id": int(sample_student_id),
    "overall_accuracy": round(float(overall_accuracy), 2),
    "average_attempts": round(float(average_attempts), 2),
    "average_hint_count": round(float(average_hint_count), 2),
    "average_response_time_ms": round(float(average_response_time), 2),
    "weak_areas": weak_areas,
    "strong_areas": strong_areas,
    "preferred_support_style": preferred_support_style
}

long_term_memory

{'student_id': 14,
 'overall_accuracy': 0.26,
 'average_attempts': 0.74,
 'average_hint_count': 1.11,
 'average_response_time_ms': 29605.34,
 'weak_areas': ['Circle Graph',
  'Equivalent Fractions',
  'Median',
  'Percent Of'],
 'strong_areas': ['Range'],
 'preferred_support_style': 'basic_explanation_support'}

In [11]:
#Create Concept-Based Memory (Concept-based memory stores concept-level history & it stores observed behavior)

concept_memory = student_df.groupby("concept_name").agg(
    total_interactions=("correct", "count"),
    correct_count=("correct", "sum"),
    total_hints=("hint_count", "sum"),
    avg_hints=("hint_count", "mean"),
    avg_attempts=("attempt_count", "mean"),
    avg_response_time_ms=("response_time_ms", "mean"),
    last_interaction_order=("interaction_order", "max")
).reset_index()

concept_memory["wrong_count"] = (
    concept_memory["total_interactions"] - concept_memory["correct_count"]
)

concept_memory["accuracy"] = (
    concept_memory["correct_count"] / concept_memory["total_interactions"]
)

def identify_behavior(row):
    if row["accuracy"] < 0.50 and row["total_interactions"] >= 3:
        return "repeated_difficulty"
    elif row["accuracy"] >= 0.75 and row["total_interactions"] >= 3:
        return "consistent_strength"
    elif row["avg_hints"] >= 2:
        return "hint_dependent"
    elif row["avg_attempts"] >= 2:
        return "needs_more_attempts"
    else:
        return "normal_progress"

concept_memory["observed_behavior"] = concept_memory.apply(identify_behavior, axis=1)

concept_memory = concept_memory.sort_values(
    by=["observed_behavior", "accuracy"],
    ascending=[True, True]
)

concept_memory.head(15)

,concept_name,total_interactions,correct_count,total_hints,avg_hints,avg_attempts,avg_response_time_ms,last_interaction_order,wrong_count,accuracy,observed_behavior
6,Range,3,3,0,0.000000,1.00,11354.666667,21617942,0,1.000000,consistent_strength
2,Finding Percents,2,0,3,1.500000,0.50,24554.500000,21617825,2,0.000000,normal_progress
5,Proportion,2,0,0,0.000000,0.00,71231.500000,21617749,2,0.000000,normal_progress
3,Median,4,0,10,2.500000,1.00,6697.250000,21617962,4,0.000000,repeated_difficulty
4,Percent Of,6,1,10,1.666667,1.00,17566.166667,21617667,5,0.166667,repeated_difficulty
0,Circle Graph,12,3,13,1.083333,0.75,32656.750000,21617825,9,0.250000,repeated_difficulty
1,Equivalent Fractions,6,2,3,0.500000,0.50,47747.333333,21617825,4,0.333333,repeated_difficulty


In [12]:
#Add Personalization Notes (This creates memory-based notes for other agents)

def generate_memory_note(row):
    if row["observed_behavior"] == "repeated_difficulty":
        return "Student has repeated difficulty in this concept. Provide simpler explanation and more examples."
    elif row["observed_behavior"] == "hint_dependent":
        return "Student often uses hints in this concept. Provide guided hints before final answers."
    elif row["observed_behavior"] == "needs_more_attempts":
        return "Student needs multiple attempts in this concept. Break tasks into smaller steps."
    elif row["observed_behavior"] == "consistent_strength":
        return "Student performs strongly in this concept. Provide less guidance or advanced practice."
    else:
        return "Student shows normal progress. Continue standard support."

concept_memory["memory_note"] = concept_memory.apply(generate_memory_note, axis=1)

concept_memory[
    [
        "concept_name",
        "total_interactions",
        "correct_count",
        "wrong_count",
        "accuracy",
        "total_hints",
        "avg_attempts",
        "observed_behavior",
        "memory_note"
    ]
].head(20)

,concept_name,total_interactions,correct_count,wrong_count,accuracy,total_hints,avg_attempts,observed_behavior,memory_note
6,Range,3,3,0,1.000000,0,1.00,consistent_strength,Student performs strongly in this concept. Pro...
2,Finding Percents,2,0,2,0.000000,3,0.50,normal_progress,Student shows normal progress. Continue standa...
5,Proportion,2,0,2,0.000000,0,0.00,normal_progress,Student shows normal progress. Continue standa...
3,Median,4,0,4,0.000000,10,1.00,repeated_difficulty,Student has repeated difficulty in this concep...
4,Percent Of,6,1,5,0.166667,10,1.00,repeated_difficulty,Student has repeated difficulty in this concep...
0,Circle Graph,12,3,9,0.250000,13,0.75,repeated_difficulty,Student has repeated difficulty in this concep...
1,Equivalent Fractions,6,2,4,0.333333,3,0.50,repeated_difficulty,Student has repeated difficulty in this concep...


In [13]:
#Generate Memory Context JSON (This is your main output for other components)
# Choose a target concept from weak areas if available,
# otherwise choose the first concept from concept memory.

if len(weak_areas) > 0:
    target_concept = weak_areas[0]
else:
    target_concept = concept_memory.iloc[0]["concept_name"]

target_row = concept_memory[
    concept_memory["concept_name"] == target_concept
].iloc[0]

memory_context = {
    "student_id": int(sample_student_id),
    "target_concept": target_concept,
    "short_term_memory": short_term_memory,
    "long_term_memory": long_term_memory,
    "concept_based_memory": {
        "concept_name": target_concept,
        "total_interactions": int(target_row["total_interactions"]),
        "correct_count": int(target_row["correct_count"]),
        "wrong_count": int(target_row["wrong_count"]),
        "accuracy": round(float(target_row["accuracy"]), 2),
        "total_hints": int(target_row["total_hints"]),
        "average_attempts": round(float(target_row["avg_attempts"]), 2),
        "observed_behavior": target_row["observed_behavior"],
        "memory_note": target_row["memory_note"]
    },
    "personalization_context_for_agents": {
        "student_support_style": preferred_support_style,
        "weak_areas": weak_areas,
        "strong_areas": strong_areas,
        "recommended_agent_usage": "Use this memory context to personalize explanation, planning, evaluation, and meta-level analysis."
    }
}

print(json.dumps(memory_context, indent=4))

{
    "student_id": 14,
    "target_concept": "Circle Graph",
    "short_term_memory": {
        "student_id": 14,
        "session_id": 263601,
        "current_concept": "Median",
        "recent_interaction_count": 7,
        "recent_accuracy": 0.43,
        "average_attempts": 1.0,
        "recent_hint_usage": 10,
        "session_status": "needs_support"
    },
    "long_term_memory": {
        "student_id": 14,
        "overall_accuracy": 0.26,
        "average_attempts": 0.74,
        "average_hint_count": 1.11,
        "average_response_time_ms": 29605.34,
        "weak_areas": [
            "Circle Graph",
            "Equivalent Fractions",
            "Median",
            "Percent Of"
        ],
        "strong_areas": [
            "Range"
        ],
        "preferred_support_style": "basic_explanation_support"
    },
    "concept_based_memory": {
        "concept_name": "Circle Graph",
        "total_interactions": 12,
        "correct_count": 3,
        "wrong_count": 9

In [14]:
#Save Memory Outputs

os.makedirs("../outputs/tables", exist_ok=True)
os.makedirs("../outputs/api_results", exist_ok=True)

concept_memory.to_csv(
    f"../outputs/tables/student_{sample_student_id}_concept_based_memory.csv",
    index=False
)

with open(f"../outputs/api_results/student_{sample_student_id}_memory_context.json", "w") as f:
    json.dump(memory_context, f, indent=4)

print("Memory outputs saved successfully.")

Memory outputs saved successfully.
